In [1]:
import os
import sys
import asyncio
import importlib.util
import nest_asyncio

# Allow asyncio.run() to work inside Jupyter
nest_asyncio.apply()

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
config_path = os.path.abspath(os.path.join(os.getcwd(), "..", "configs", "config.py"))
spec = importlib.util.spec_from_file_location("local_config", config_path)
config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config)

from arxiv_scholar.retrieval.orchestrator import AdvancedRetriever


/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize Orchestrator
# NOTE: Make sure OPENROUTER_API_KEY is exported in your environment!
retriever = AdvancedRetriever(
    collection_name="arxiv_papers",
    qdrant_host=config.QDRANT_HOST,
    qdrant_port=config.QDRANT_PORT
)


/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 13.50it/s]


In [3]:
def print_results(query: str, results: list):
    print(f"\n{'='*80}")
    print(f"QUERY: {query}")
    if results:
        path = results[0].get('_query_path', 'unknown')
        print(f"ROUTE TAKEN: {path.upper()}")
    print(f"{'='*80}\n")
    
    for i, res in enumerate(results[:3]):
        print(f"Rank {i+1} [Score: {res['score']:.4f}]")
        print(f"Text Snippet: {res['text'][:200]}...\n")


### Test 1: DIRECT Search
Long, semantically rich queries should bypass the LLM entirely and return instantly.

In [6]:
long_query = "What is difference between contrastive and incremental learning?"
results_direct = asyncio.run(retriever.retrieve(long_query, limit=5))
print_results(long_query, results_direct)


No LLM client configured for DECOMPOSE. Falling back to DIRECT.



QUERY: What is difference between contrastive and incremental learning?
ROUTE TAKEN: DECOMPOSE

Rank 1 [Score: 0.5833]
Text Snippet: Task and function vectors from contrastive prompts. The closest methodological precedent is the line of work that constructs task representations from differences between contrastive prompts. Incontex...

Rank 2 [Score: 0.5000]
Text Snippet: Eysenbach, B., Zhang, T., Levine, S., and Salakhutdinov, R. R. Contrastive learning as goal-conditioned reinforcement learning. Advances in Neural Information Processing Systems , 35:35603-35620, 2022...

Rank 3 [Score: 0.5000]
Text Snippet: In this section, we describe the related work, covering the concept drift in malware detection, various approaches including contrastive learning and other incremental learning methods to address conc...



### Test 2: HyDE Search
Short, sparse queries lack contextual signal. They should be intercepted by the router (length <= 4) and hit the LLM to generate a hypothetical academic abstract before running the hybrid search.

In [5]:
short_query = "attention mechanism"
results_hyde = asyncio.run(retriever.retrieve(short_query, limit=5))
print_results(short_query, results_hyde)



QUERY: attention mechanism
ROUTE TAKEN: HYDE

Rank 1 [Score: 0.5000]
Text Snippet: This section investigates the answer to RQ 5. In Table 7, we compared performance for GITE with two different attention mechanisms: GAT [72] and the attention mechanism of Transformer based on query a...

Rank 2 [Score: 0.5000]
Text Snippet: The attention substep is...

Rank 3 [Score: 0.3333]
Text Snippet: the attention substep is...

